## 5.1 Evaluating generative text models

### 5.1.1 Using GPT to generate text

In [2]:
# 使用 colab 需要每次上传 chapter04.py 到colab服务器。修改服务器中文件后需要重启python kernel才能读取更新后的文件
import torch
from chapter04 import GPTModel

In [3]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,      # context_length 改为256，以便在笔记本上进行训练
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features

```torch.unsqueeze(input, dim) → Tensor```

返回一个在指定位置插入了一个维度大小为1的新张量。    
返回的张量与这个张量共享相同的基本数据。

```
>>> x = torch.tensor([1, 2, 3, 4])
>>> torch.unsqueeze(x, 0)
tensor([[ 1,  2,  3,  4]])
>>> torch.unsqueeze(x, 1)
tensor([[ 1],
        [ 2],
        [ 3],
        [ 4]])
```

在深度学习中：   
**输入 = batch**

所以必须是：   
**(batch, sequence)**

维度变化： 一维tensor `(seq_len,)` 执行 `.unsqueeze(0)` 后变成 `(1, seq_len)`

In [4]:
# unsqueeze 作用是解序列，把数据变成更高一维的数据，原数据作为新数据的列或者行
test = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(test.shape)
unsqu_test = test.unsqueeze(0)
print(unsqu_test, unsqu_test.shape)

print()
test = torch.tensor([[1 ,2, 3, 4, 5, 6]])
print(test.shape)
squ_test = test.squeeze(0)
print(squ_test, squ_test.shape)

torch.Size([2, 3])
tensor([[[1, 2, 3],
         [4, 5, 6]]]) torch.Size([1, 2, 3])

torch.Size([1, 6])
tensor([1, 2, 3, 4, 5, 6]) torch.Size([6])


In [5]:
import tiktoken
from chapter04 import generate_text_simple

# 把 text 用分词器转换为token ids，返回batch化后的tensor
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'}) # list[int]
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # 1D (seq_len,) -> 2D (1, seq_len)， batch size =1
    return encoded_tensor

# token ids转换为文本后序列化并返回
def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model = model,
    idx = text_to_token_ids(start_context, tokenizer),
    max_new_tokens = 10,
    context_size = GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you rentingetic wasnم refres RexMeCHicular stren


### 5.1.2 Calculating the text generation loss

![from input text to output text](https://raw.githubusercontent.com/ipdor/Pictures/master/20260425144904485.png)

用两个输入示例（"every effort moves" 和 "I really like"）来演示输入到输出文本的大概过程

In [6]:
inputs = torch.tensor([[16833, 3626, 6100],     # ["every effort moves",
                       [40, 1107, 588]])        # "I really like"]

目标是生成下面的两组target数据，注意 target 是inputs向后移动一个字符的内容

In [7]:
targets = torch.tensor([[3626, 6100, 345 ],     # [" effort moves you",
                        [1107, 588, 11311]])    # " really like chocolate"]

In [8]:
with torch.no_grad():
    logits = model(inputs)
probas = torch.softmax(logits, dim=-1)          # 对应图5.4, step 2 将logits转换为概率probability scores
print(probas.shape) # [batch_size, num_tokens, embedding_dim] = [2, 3, 50257]

torch.Size([2, 3, 50257])


In [9]:
# 对应图5.4, step 3 and 4 对概率调用argmax获取预测的的token ids
token_ids = torch.argmax(probas, dim=-1, keepdim=True) 
print("Token IDs:\n", token_ids)

Token IDs:
 tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [41751]]])


In [10]:
# 对应图5.4, step 5 将 index 映射回文本text
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Outputs batch 1:"
f" {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

Targets batch 1:  effort moves you
Outputs batch 1:  Armed heNetflix


#### 手工实现计算交叉熵损失，实现模型评估

评估过程：   
1. 通过比较生成的文本和目标文本，用定量的方式确定生成token的质量，表示它们之间的"距离"；   
2. 后期使用的训练函数将会使用这种"距离"来更新模型权重，实现优化的目的。

![Calculating the loss ](https://raw.githubusercontent.com/ipdor/Pictures/master/20260425171205961.png)


In [11]:
test = torch.tensor([[1, 2, 3], 
                    [4, 5, 6], 
                    [7, 8, 9]])
sliced = test[[0,1,2], [0,1,2]] # 按顺序组合，取test(0, 0), (1, 1), (2, 2)
print(sliced)

tensor([1, 5, 9])


In [12]:
# probas [batch_size, num_tokens, embedding_dim] = [2, 3, 50257]
# target [batch_size, num_tokens] = [2, 3]
text_idx = 0
print(targets[text_idx])  # 
target_probas_1 = probas[text_idx, [0,1,2]]
print("Text 1:", target_probas_1)   # 取 probas[0, [0,1,2]]， 因为刚好只有3行，也就是第一批里面所有内容，等价 probas[0]

# 切片第二维 [0,1,2]                                        三个 token 的位置索引
# 切片第三维 targets[text_idx] = ([3626, 6100,  345])       三个位置的正确 token ID !
# 搭配第0维 text_idx=0                                      第一批token
# 组合起来就是 [0, 0, 3626] [0, 1, 6100]  [0, 2, 345] 取probas[0] 第0行的3626列，第一行的6100列，第三行的345列
# 分别是输出第一批中，正确的第一个字符、第二个字符、第三个字符对应的概率
print(probas[text_idx, 0, 3626], probas[text_idx, 1, 6100], probas[text_idx, 2, 345])
print(probas[text_idx, [0,1,2], targets[text_idx]])

tensor([3626, 6100,  345])
Text 1: tensor([[1.8849e-05, 1.5172e-05, 1.1687e-05,  ..., 2.2409e-05, 6.9776e-06,
         1.8776e-05],
        [9.1569e-06, 1.0062e-05, 7.8786e-06,  ..., 2.9090e-05, 6.0103e-06,
         1.3571e-05],
        [2.9877e-05, 8.8507e-06, 1.5741e-05,  ..., 3.5456e-05, 1.4094e-05,
         1.3526e-05]])
tensor(7.4540e-05) tensor(3.1061e-05) tensor(1.1563e-05)
tensor([7.4540e-05, 3.1061e-05, 1.1563e-05])


图 5.7 第一到三步：

In [13]:
# 打印训练前和目标标记相对应的初始 softmax 概率分数（从模型输出的概率分布中，取出“正确 token 的概率”)
# [0,1,2]               三个 token 的位置索引
# targets[text_idx]     三个位置的正确 token ID !
text_idx = 0
target_probas_1 = probas[text_idx, [0,1,2], targets[text_idx]]
print("Text 1:", target_probas_1)

text_idx = 1
target_probas_2 = probas[text_idx, [0,1,2], targets[text_idx]]
print("Text 2:", target_probas_2)

Text 1: tensor([7.4540e-05, 3.1061e-05, 1.1563e-05])
Text 2: tensor([1.0337e-05, 5.6776e-05, 4.7559e-06])


图 5.7 第四步：

In [14]:
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
print(log_probas)

tensor([ -9.5042, -10.3796, -11.3677, -11.4798,  -9.7764, -12.2561])


In [15]:
avg_log_probas = torch.mean(log_probas)
print(avg_log_probas)

tensor(-10.7940)


交叉熵损失 (***cross entropy* loss**) : 把平均对数概率average log probability变为负的平均对数概率negative average log probability的操作。

核心上，交叉熵损失是机器学习和深度学习中一种流行的度量方法，用于衡量两个概率分布之间的差异——通常是标签的真实分布（在这里，是数据集中的标记）和来自模型的预测分布（例如，LLM 生成的标记概率）。

In [16]:
neg_avg_log_probas = avg_log_probas * -1
print(neg_avg_log_probas)

tensor(10.7940)


#### 调用框架实现计算交叉熵损失

In [17]:
print("Logits shape:", logits.shape)
print("Targets shape:", targets.shape)

Logits shape: torch.Size([2, 3, 50257])
Targets shape: torch.Size([2, 3])


In [18]:
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()
print("Flattened logits:", logits_flat.shape)
print("Flattened targets:", targets_flat.shape)

Flattened logits: torch.Size([6, 50257])
Flattened targets: torch.Size([6])


Pytorch 支持直接计算出负的平均对数概率negative average log probability，省去调用 `softmax` 计算概率、选择与target ID 相对应的概率分数、计算负平均对数概率这些步骤。


In [19]:
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(loss)

tensor(10.7940)


### 5.1.3 Calculating the training and validation set losses

In [22]:
import os
import urllib.request


if not os.path.exists("the-verdict.txt"):
    url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch02/01_main-chapter-code/the-verdict.txt"
    file_path = "the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)
    print("Fiel Downloaded")

Fiel Downloaded


In [24]:
file_path = "the-verdict.txt"
with open(file_path, "r", encoding="utf-8") as file:
    text_data = file.read()
print(text_data[:200])

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a


In [26]:
total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))
print("Characters:", total_characters)
print("Tokens:", total_tokens)

Characters: 20479
Tokens: 5145


In [ ]:
train_ratio = 0.9
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]
print("split_idx:", split_idx)

split_idx: 18431


In [42]:
from chapter04 import create_dataloader_v1
torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size = 2,
    max_length = GPT_CONFIG_124M["context_length"],
    stride = GPT_CONFIG_124M["context_length"],
    drop_last = True,
    shuffle = True,
    num_workers = 0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size = 2,
    max_length = GPT_CONFIG_124M["context_length"],
    stride = GPT_CONFIG_124M["context_length"],
    drop_last = False,
    shuffle = False,
    num_workers = 0
)

In [43]:
print("Train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)

print("\nValidation loader:")
for x, y in val_loader:
    print(x.shape, y.shape)

Train loader:
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])

Validation loader:
torch.Size([2, 256]) torch.Size([2, 256])


```torch.flatten(input, start_dim=0, end_dim=-1) → Tensor```


将input重塑为一维张量。如果传递了start_dim或end_dim，则仅从start_dim开始到end_dim结束的维度被压平。元素在input中的顺序保持不变。

In [76]:
n = torch.rand(2,3,4) # [batch_size, seq_len, emb_dim]

print(n, n.shape)
# print(n.flatten()) # 直接调用是转换为1维[1,n]
print(n.flatten(0,1).shape)   # flatten 前2个维度，相当于每个batch连在第一个batch的下面 [6, 4]

# n.flatten(0,1)不等价于下面的stack，等价于下面的cat
print(torch.stack((n[0], n[1]), dim=0)) # 新增一个维度再拼接 2 × [3,4] -> [2, 3, 4]
print(torch.cat((n[0], n[1]), dim=0))   # 沿已有维度拼接 2 × [3,4] -> [6, 4]

tensor([[[0.5448, 0.5481, 0.0334, 0.6258],
         [0.9622, 0.9571, 0.7815, 0.9591],
         [0.0159, 0.4674, 0.9610, 0.0349]],

        [[0.0428, 0.0949, 0.2674, 0.6335],
         [0.5072, 0.1678, 0.6436, 0.1264],
         [0.0024, 0.5600, 0.8421, 0.0182]]]) torch.Size([2, 3, 4])
torch.Size([6, 4])
tensor([[[0.5448, 0.5481, 0.0334, 0.6258],
         [0.9622, 0.9571, 0.7815, 0.9591],
         [0.0159, 0.4674, 0.9610, 0.0349]],

        [[0.0428, 0.0949, 0.2674, 0.6335],
         [0.5072, 0.1678, 0.6436, 0.1264],
         [0.0024, 0.5600, 0.8421, 0.0182]]])
tensor([[0.5448, 0.5481, 0.0334, 0.6258],
        [0.9622, 0.9571, 0.7815, 0.9591],
        [0.0159, 0.4674, 0.9610, 0.0349],
        [0.0428, 0.0949, 0.2674, 0.6335],
        [0.5072, 0.1678, 0.6436, 0.1264],
        [0.0024, 0.5600, 0.8421, 0.0182]])


In [111]:
def calc_loss_batch(input_batch, output_batch, model, device):
    # [2, 256]
    input_batch = input_batch.to(device)
    output_batch = output_batch.to(device)
    # [2, 256, 50257]
    logits = model(input_batch)
    # [N=512, C=50257], [N=512] 要求除了C之外其他维度完全匹配
    loss = torch.nn.functional.cross_entropy(
        logits.flatten(0,1), output_batch.flatten()
    )
    return loss

In [112]:
# 测试 calc_loss_batch
input_b = torch.randint(0, 9, (2,256))
output_b = torch.randint(0, 9, (2,256))

calc_loss_batch(input_b, output_b, model, "cpu")

tensor(10.8361, grad_fn=<NllLossBackward0>)

In [ ]:
# 遍历计算给定数据加载器中的所有批次的loss, 最后返回平均loss
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0
    if len(data_loader)==0:
        return float("Nan")
    elif num_batches==None:
        num_batches = len(data_loader) # 如果未指定批数，使用所有数据
    else:
        # 如果 `num_batches` 超过了数据加载器中的批次数，则减少批次数以匹配实际所有的总批次数
        num_batches = min(len(data_loader), num_batches)
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i >= num_batches:
            break
        loss = calc_loss_batch(input_batch, target_batch, model, device)
        total_loss += loss.item()
    return total_loss / num_batches

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

with torch.no_grad(): # 尚未开始训练，为了提高效率，禁用梯度追踪
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: 10.98758347829183
Validation loss: 10.98110580444336


## 5.2 Training an LLM

![Train loop](https://raw.githubusercontent.com/ipdor/Pictures/master/20260426172033736.png)

图 5.11 中的流程图描绘了一个典型的 PyTorch 神经网络训练工作流，我们将其用于训练 LLM。它概述了八个步骤，从迭代每个 epoch 开始，接着是处理批次、重置梯度、计算损失和新梯度、更新权重，最后以打印损失和生成文本样本等监控步骤结束。

1
